# Notebook 00 — Construção do Cache openSMILE por Bloco

Este notebook precisa ser executado **antes** dos notebooks 1, 2 e 3.  
Ele constrói e salva `Dados/comparative_outputs/common/opensmile_blocks.parquet`.

## Fontes de dados

| Arquivo | Descrição |
|---|---|
| `Dados/features/{song_id}.csv` | Features openSMILE frame-level (sep=`;`, coluna `frameTime` a cada 0.5s) |
| `Dados/valence.csv` | Anotações de valence DEAM — formato wide: `song_id \| sample_15000ms \| sample_15500ms \| ...` |
| `Dados/arousal.csv` | Anotações de arousal DEAM — mesmo formato |
| `Dados/fingerprint_rank/fingerprint_rank.parquet` | Define os blocos canônicos (janelas de 10s, `block_start_sec`/`block_end_sec`) |

## O que este notebook faz

1. Carrega os rótulos DEAM (wide → long) e calcula `valence`/`arousal` médio por bloco de 10s.
2. Valida que esses valores batem com os do parquet de fingerprint.
3. Para cada bloco, agrega 260 descritores openSMILE por média e desvio-padrão, totalizando 520 atributos preditores.
4. Audita a taxonomia de Panda pelo núcleo acústico de cada descritor, sem tratar prefixos, estatísticas ou derivadas como categorias.
5. Salva o resultado como `opensmile_blocks.parquet`.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import re
import time
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── CAMINHOS ───────────────────────────────────────────────────────────────
DATA_DIR       = Path("Dados")
FEATURES_DIR   = DATA_DIR / "features"
VALENCE_PATH   = DATA_DIR / "valence.csv"
AROUSAL_PATH   = DATA_DIR / "arousal.csv"
FP_RANK_PATH   = DATA_DIR / "fingerprint_rank" / "fingerprint_rank.parquet"

COMMON_DIR     = DATA_DIR / "comparative_outputs" / "common"
OUTPUT_PATH    = COMMON_DIR / "opensmile_blocks.parquet"
FOLD_PATH      = COMMON_DIR / "fold_assignments.csv"  # criado pelos notebooks 1-3

COMMON_DIR.mkdir(parents=True, exist_ok=True)

# ── CONSTANTES ─────────────────────────────────────────────────────────────
SONG_ID_COL   = "song_id"
BLOCK_ID_COL  = "block_id"
START_COL     = "block_start_sec"
END_COL       = "block_end_sec"
VALENCE_COL   = "valence"
AROUSAL_COL   = "arousal"
QUADRANT_COL  = "quadrant_label"
METHOD        = "opensmile"

# Prefixos de feature — mesma lógica que fingerprint_band_rank usa mean/std por banda
FEATURE_PREFIX = "os_"       # prefixo comum para detecção de todas as colunas de feature
MEAN_PREFIX    = "os_mean__" # média dos frames no bloco
STD_PREFIX     = "os_std__"  # desvio padrão dos frames no bloco

# Verifica existência dos arquivos de entrada
for p in [FEATURES_DIR, VALENCE_PATH, AROUSAL_PATH, FP_RANK_PATH]:
    assert p.exists(), f"Arquivo/pasta não encontrado: {p}"

print("Setup OK")
print(f"  features dir  : {FEATURES_DIR}")
print(f"  valence.csv   : {VALENCE_PATH}")
print(f"  arousal.csv   : {AROUSAL_PATH}")
print(f"  fingerprint   : {FP_RANK_PATH}")
print(f"  output        : {OUTPUT_PATH}")
print(f"  prefixo mean  : {MEAN_PREFIX}")
print(f"  prefixo std   : {STD_PREFIX}")

Setup OK
  features dir  : Dados\features
  valence.csv   : Dados\valence.csv
  arousal.csv   : Dados\arousal.csv
  fingerprint   : Dados\fingerprint_rank\fingerprint_rank.parquet
  output        : Dados\comparative_outputs\common\opensmile_blocks.parquet
  prefixo mean  : os_mean__
  prefixo std   : os_std__


## 1. Carregar rótulos DEAM (valence.csv / arousal.csv)

Formato wide: `song_id | sample_15000ms | sample_15500ms | ...` (anotações a cada 500ms, começando em 15s).

In [2]:
def load_deam_labels(path: Path, label_col: str) -> pd.DataFrame:
    """
    Lê valence.csv ou arousal.csv (formato wide) e converte para long:
    song_id | time_sec | {label_col}
    """
    df_wide = pd.read_csv(path)
    sample_cols = [c for c in df_wide.columns if c.startswith("sample_")]

    df_long = df_wide.melt(
        id_vars=[SONG_ID_COL],
        value_vars=sample_cols,
        var_name="sample_col",
        value_name=label_col,
    )
    # Extrai tempo em segundos: "sample_15000ms" → 15.0
    df_long["time_sec"] = (
        df_long["sample_col"]
        .str.extract(r"sample_(\d+)ms", expand=False)
        .astype(float) / 1000.0
    )
    df_long = (
        df_long[[SONG_ID_COL, "time_sec", label_col]]
        .dropna()
        .sort_values([SONG_ID_COL, "time_sec"])
        .reset_index(drop=True)
    )
    return df_long


v_long = load_deam_labels(VALENCE_PATH, VALENCE_COL)
a_long = load_deam_labels(AROUSAL_PATH, AROUSAL_COL)

print(f"Valence long: {v_long.shape}  |  time range: {v_long['time_sec'].min():.1f}s – {v_long['time_sec'].max():.1f}s")
print(f"Arousal long: {a_long.shape}  |  time range: {a_long['time_sec'].min():.1f}s – {a_long['time_sec'].max():.1f}s")
print(f"Songs com valence: {v_long[SONG_ID_COL].nunique()}  |  Songs com arousal: {a_long[SONG_ID_COL].nunique()}")
print(f"\nAmostra valence (song_id=2):")
print(v_long[v_long[SONG_ID_COL] == 2].head(6).to_string(index=False))

Valence long: (129998, 3)  |  time range: 15.0s – 626.0s
Arousal long: (129999, 3)  |  time range: 15.0s – 626.5s
Songs com valence: 1802  |  Songs com arousal: 1802

Amostra valence (song_id=2):
 song_id  time_sec   valence
       2      15.0 -0.073341
       2      15.5 -0.074661
       2      16.0 -0.074077
       2      16.5 -0.078154
       2      17.0 -0.081588
       2      17.5 -0.080873


## 2. Carregar blocos canônicos do fingerprint

O `fingerprint_rank.parquet` define quais blocos existem e já contém `valence`/`arousal` calculados como média das anotações DEAM dentro da janela do bloco.

In [3]:
blocks = pd.read_parquet(
    FP_RANK_PATH,
    columns=[SONG_ID_COL, BLOCK_ID_COL, START_COL, END_COL,
             VALENCE_COL, AROUSAL_COL, QUADRANT_COL],
)
print(f"Blocos canônicos: {blocks.shape}")
print(f"Músicas únicas  : {blocks[SONG_ID_COL].nunique()}")
print(f"Blocos por música (média): {len(blocks) / blocks[SONG_ID_COL].nunique():.1f}")
print(f"\nDistribuição de quadrantes:")
print(blocks[QUADRANT_COL].value_counts().to_string())
print(f"\nAmostra (song_id=2):")
print(blocks[blocks[SONG_ID_COL] == 2].to_string(index=False))

Blocos canônicos: (6506, 7)
Músicas únicas  : 1802
Blocos por música (média): 3.6

Distribuição de quadrantes:
quadrant_label
Alegre/Energetico     3217
Triste/Melancolico    1382
Tenso/Raivoso         1019
Calmo/Relaxado         888

Amostra (song_id=2):
 song_id  block_id  block_start_sec  block_end_sec   valence   arousal     quadrant_label
       2         0             15.0           25.0 -0.085140 -0.137974 Triste/Melancolico
       2         1             25.0           35.0 -0.241535 -0.191833 Triste/Melancolico
       2         2             35.0           45.0 -0.319857 -0.262745 Triste/Melancolico


## 3. Validar rótulos: DEAM vs fingerprint parquet

Verifica que a valence/arousal do parquet bate com a média das anotações DEAM na janela do bloco.

In [4]:
def compute_block_labels_from_deam(
    blocks: pd.DataFrame,
    v_long: pd.DataFrame,
    a_long: pd.DataFrame,
    n_songs: int = 20,
) -> pd.DataFrame:
    """
    Para uma amostra de músicas, recalcula valence/arousal do bloco
    como média das anotações DEAM dentro de [block_start_sec, block_end_sec).
    """
    labels = v_long.merge(a_long, on=[SONG_ID_COL, "time_sec"])
    sample_songs = blocks[SONG_ID_COL].unique()[:n_songs]
    rows = []

    for sid in sample_songs:
        song_labels = labels[labels[SONG_ID_COL] == sid]
        if song_labels.empty:
            continue
        for _, blk in blocks[blocks[SONG_ID_COL] == sid].iterrows():
            mask = (
                (song_labels["time_sec"] >= blk[START_COL]) &
                (song_labels["time_sec"] < blk[END_COL])
            )
            window = song_labels[mask]
            if window.empty:
                continue
            rows.append({
                SONG_ID_COL: sid,
                BLOCK_ID_COL: blk[BLOCK_ID_COL],
                START_COL: blk[START_COL],
                "valence_fp":   blk[VALENCE_COL],
                "arousal_fp":   blk[AROUSAL_COL],
                "valence_deam": window[VALENCE_COL].mean(),
                "arousal_deam": window[AROUSAL_COL].mean(),
                "n_samples":    len(window),
            })

    return pd.DataFrame(rows)


validation = compute_block_labels_from_deam(blocks, v_long, a_long, n_songs=30)

v_corr = validation["valence_fp"].corr(validation["valence_deam"])
a_corr = validation["arousal_fp"].corr(validation["arousal_deam"])
v_mae  = (validation["valence_fp"] - validation["valence_deam"]).abs().mean()
a_mae  = (validation["arousal_fp"] - validation["arousal_deam"]).abs().mean()

print("Validação rótulos fingerprint parquet vs DEAM annotations:")
print(f"  Valence  — Pearson: {v_corr:.6f}  MAE: {v_mae:.6f}")
print(f"  Arousal  — Pearson: {a_corr:.6f}  MAE: {a_mae:.6f}")
print(f"  Blocos validados: {len(validation)}")

if v_corr > 0.999 and a_corr > 0.999:
    print("\n✓ Rótulos do fingerprint parquet são consistentes com DEAM.")
else:
    print("\n⚠ Divergência detectada — verifique o alinhamento.")
    print(validation.head(10).to_string())

Validação rótulos fingerprint parquet vs DEAM annotations:
  Valence  — Pearson: 1.000000  MAE: 0.000000
  Arousal  — Pearson: 1.000000  MAE: 0.000000
  Blocos validados: 90

✓ Rótulos do fingerprint parquet são consistentes com DEAM.


## 4. Construir features openSMILE por bloco

Para cada bloco canônico:
- Carrega `Dados/features/{song_id}.csv` (sep=`;`, `frameTime` a cada 0.5s)
- Seleciona frames dentro de `[block_start_sec, block_end_sec)`
- Calcula a **média** e o **desvio-padrão** de cada descritor openSMILE no bloco
- Exclui `frameTime`: 260 descritores numéricos × 2 agregações = **520 atributos preditores**
- Fallback: frame mais próximo se nenhum frame cair na janela

In [5]:
def read_opensmile_csv(song_id: int) -> Optional[pd.DataFrame]:
    path = FEATURES_DIR / f"{song_id}.csv"
    if not path.exists():
        return None
    try:
        df = pd.read_csv(path, sep=";")
    except Exception:
        return None
    if "frameTime" not in df.columns:
        return None
    return df


def aggregate_frames_for_block(
    osdf: pd.DataFrame,
    feat_cols: List[str],
    start: float,
    end: float,
) -> Tuple[dict, dict]:
    """
    Agrega os frames openSMILE dentro de [start, end).
    Retorna (mean_dict, std_dict) — mesma lógica do fingerprint_band_rank
    que calcula mag_mean_* e mag_std_* para cada banda.
    Se nenhum frame cair na janela, usa o frame mais próximo (std=0).
    """
    mask = (osdf["frameTime"] >= start) & (osdf["frameTime"] < end)
    frames = osdf.loc[mask, feat_cols]
    if len(frames) == 0:
        # Fallback: frame mais próximo do início do bloco
        nearest = (osdf["frameTime"] - start).abs().idxmin()
        frames = osdf.loc[[nearest], feat_cols]
    mean_dict = frames.mean().to_dict()
    # ddof=0 para consistência com np.nanstd do fingerprint; fillna(0) no fallback de 1 frame
    std_dict  = frames.std(ddof=0).fillna(0.0).to_dict()
    return mean_dict, std_dict


def compute_quadrant(valence: float, arousal: float):
    """Classifica quadrante emocional — mesma convenção do fingerprint_band_rank."""
    if pd.isna(valence) or pd.isna(arousal):
        return "INDEFINIDO", "INDEFINIDO"
    v, a = float(valence), float(arousal)
    if v >= 0 and a >= 0:
        return "Q1", "Q1_High_Valence_High_Arousal"
    elif v < 0 and a >= 0:
        return "Q2", "Q2_Low_Valence_High_Arousal"
    elif v < 0 and a < 0:
        return "Q3", "Q3_Low_Valence_Low_Arousal"
    else:
        return "Q4", "Q4_High_Valence_Low_Arousal"


# Priorizar explicitamente Dados/features/2.csv, usado na auditoria metodológica.
feat_cols_ref = None
candidate_song_ids = list(dict.fromkeys([2] + [int(s) for s in sorted(blocks[SONG_ID_COL].unique())]))
for sid in candidate_song_ids:
    osdf_sample = read_opensmile_csv(sid)
    if osdf_sample is not None:
        feat_cols_ref = [
            c for c in osdf_sample.columns
            if c != "frameTime" and pd.api.types.is_numeric_dtype(osdf_sample[c])
        ]
        print(f"Colunas openSMILE detectadas (song_id={sid}): {len(feat_cols_ref)} features")
        print(f"  Primeiras: {feat_cols_ref[:5]}")
        print(f"  frameTime range: {osdf_sample['frameTime'].min():.1f} – {osdf_sample['frameTime'].max():.1f}s")
        print(f"  n frames: {len(osdf_sample)}")
        print(f"  → por bloco: {len(feat_cols_ref)} mean ({MEAN_PREFIX}) + {len(feat_cols_ref)} std ({STD_PREFIX}) = {len(feat_cols_ref)*2} features")
        break

assert feat_cols_ref is not None, "Nenhum arquivo openSMILE válido encontrado em Dados/features/"

Colunas openSMILE detectadas (song_id=2): 260 features
  Primeiras: ['F0final_sma_stddev', 'F0final_sma_amean', 'voicingFinalUnclipped_sma_stddev', 'voicingFinalUnclipped_sma_amean', 'jitterLocal_sma_stddev']
  frameTime range: 0.0 – 224.0s
  n frames: 449
  → por bloco: 260 mean (os_mean__) + 260 std (os_std__) = 520 features


In [6]:
print("=" * 65)
print("CONSTRUINDO CACHE openSMILE POR BLOCO")
print("=" * 65)
print(f"Blocos alvo   : {len(blocks)}")
print(f"Músicas alvo  : {blocks[SONG_ID_COL].nunique()}")
print(f"Features base : {len(feat_cols_ref)} → {len(feat_cols_ref)*2} por bloco (mean + std)")
print()

t_global = time.time()
parts = []
stats = {"songs_ok": 0, "songs_missing": 0, "blocks_fallback": 0, "blocks_ok": 0}

songs = sorted(blocks[SONG_ID_COL].unique())
for i, sid in enumerate(songs):
    osdf = read_opensmile_csv(sid)
    if osdf is None:
        stats["songs_missing"] += 1
        continue

    stats["songs_ok"] += 1
    song_blocks = blocks[blocks[SONG_ID_COL] == sid]

    for _, blk in song_blocks.iterrows():
        start, end = blk[START_COL], blk[END_COL]

        n_frames_in_window = int(
            ((osdf["frameTime"] >= start) & (osdf["frameTime"] < end)).sum()
        )
        if n_frames_in_window == 0:
            stats["blocks_fallback"] += 1
        else:
            stats["blocks_ok"] += 1

        mean_agg, std_agg = aggregate_frames_for_block(osdf, feat_cols_ref, start, end)

        v_val = float(blk[VALENCE_COL])
        a_val = float(blk[AROUSAL_COL])
        quadrant, quadrant_label = compute_quadrant(v_val, a_val)
        row = {
            SONG_ID_COL:          int(blk[SONG_ID_COL]),
            "filename":           f"{int(blk[SONG_ID_COL])}.csv",
            BLOCK_ID_COL:         int(blk[BLOCK_ID_COL]),
            START_COL:            float(start),
            END_COL:              float(end),
            "block_duration_sec": float(end - start),
            VALENCE_COL:          v_val,
            AROUSAL_COL:          a_val,
            "quadrant":           quadrant,
            QUADRANT_COL:         quadrant_label,
            "method":             METHOD,
            "n_frames_in_window": n_frames_in_window,
        }
        for feat, val in mean_agg.items():
            row[f"{MEAN_PREFIX}{feat}"] = float(val) if pd.notna(val) else float("nan")
        for feat, val in std_agg.items():
            row[f"{STD_PREFIX}{feat}"] = float(val) if pd.notna(val) else float("nan")
        parts.append(row)

    if (i + 1) % 200 == 0 or (i + 1) == len(songs):
        elapsed = time.time() - t_global
        print(f"  {i + 1:4d}/{len(songs)} músicas | {len(parts):6d} blocos | {elapsed:.1f}s")

print(f"\nLoop concluído em {time.time() - t_global:.1f}s")
print(f"  Músicas processadas : {stats['songs_ok']}")
print(f"  Músicas sem CSV     : {stats['songs_missing']}")
print(f"  Blocos normais      : {stats['blocks_ok']}")
print(f"  Blocos fallback     : {stats['blocks_fallback']} (frame mais próximo, std=0)")

CONSTRUINDO CACHE openSMILE POR BLOCO
Blocos alvo   : 6506
Músicas alvo  : 1802
Features base : 260 → 520 por bloco (mean + std)



   200/1802 músicas |    600 blocos | 4.7s


   400/1802 músicas |   1200 blocos | 9.4s


   600/1802 músicas |   1800 blocos | 14.6s


   800/1802 músicas |   2400 blocos | 18.9s


  1000/1802 músicas |   3000 blocos | 22.0s


  1200/1802 músicas |   3600 blocos | 25.0s


  1400/1802 músicas |   4200 blocos | 28.1s


  1600/1802 músicas |   4800 blocos | 31.2s


  1800/1802 músicas |   6463 blocos | 36.7s
  1802/1802 músicas |   6506 blocos | 36.8s

Loop concluído em 36.8s
  Músicas processadas : 1802
  Músicas sem CSV     : 0
  Blocos normais      : 6506
  Blocos fallback     : 0 (frame mais próximo, std=0)


## 5. Montar DataFrame e imputar NaN

In [7]:
result = pd.DataFrame(parts)
os_mean_cols = [c for c in result.columns if c.startswith(MEAN_PREFIX)]
os_std_cols  = [c for c in result.columns if c.startswith(STD_PREFIX)]
os_cols      = os_mean_cols + os_std_cols

print(f"Shape antes da imputação: {result.shape}")
nan_counts = result[os_cols].isna().sum()
cols_with_nan = (nan_counts > 0).sum()
total_nan = nan_counts.sum()
print(f"Colunas com NaN: {cols_with_nan} de {len(os_cols)}")
print(f"Total NaN em features: {total_nan}")

# Imputação por média global da coluna
if total_nan > 0:
    col_means = result[os_cols].mean()
    result[os_cols] = result[os_cols].fillna(col_means)
    remaining_nan = result[os_cols].isna().sum().sum()
    print(f"NaN após imputação: {remaining_nan}")
else:
    print("Sem NaN — imputação não necessária.")

print(f"\nShape final: {result.shape}")
print(f"  Features mean ({MEAN_PREFIX}): {len(os_mean_cols)}")
print(f"  Features std  ({STD_PREFIX}) : {len(os_std_cols)}")
print(f"  Total features              : {len(os_cols)}")
print(f"  Músicas únicas: {result[SONG_ID_COL].nunique()}")
print(f"  Blocos únicos : {len(result)}")
print(f"\nDistribuição n_frames_in_window:")
print(result["n_frames_in_window"].describe().to_string())

Shape antes da imputação: (6506, 532)
Colunas com NaN: 0 de 520
Total NaN em features: 0
Sem NaN — imputação não necessária.

Shape final: (6506, 532)
  Features mean (os_mean__): 260
  Features std  (os_std__) : 260
  Total features              : 520
  Músicas únicas: 1802
  Blocos únicos : 6506

Distribuição n_frames_in_window:
count    6506.000000
mean       19.656625
std         0.891233
min         8.000000
25%        20.000000
50%        20.000000
75%        20.000000
max        20.000000


## 6. Auditar a taxonomia de Panda para as features openSMILE

A classificação é feita pelo **descritor acústico original**, não pelos artefatos de nomenclatura do pipeline:

- `frameTime` é referência temporal e não entra como feature preditora;
- os prefixos `os_mean__` e `os_std__` indicam somente a agregação no bloco;
- os sufixos `_amean` e `_stddev` são funcionais do arquivo openSMILE;
- `_de` identifica uma derivada do descritor, mas não transforma automaticamente a feature em `Rhythm / Temporal`;
- o núcleo `MFCC`, inclusive em `os_std__pcm_fftMag_mfcc_sma_de[1]_amean`, permanece em `Timbre / Tone Colour`.

A tabela soma separadamente os **260 descritores frame-level** e os **520 atributos por bloco**. Os CSVs detalhado e resumido, além da figura, são salvos em `Dados/comparative_outputs/common/`.

In [8]:
PANDA_CATEGORY_ORDER = [
    "Pitch / Melody",
    "Dynamics",
    "Harmony",
    "Timbre / Tone Colour",
    "Rhythm / Temporal",
    "Musical Texture",
    "Musical Form",
    "Expressivity",
]

PANDA_CATEGORY_COLORS = {
    "Pitch / Melody": "#2f6f9f",
    "Dynamics": "#11a579",
    "Harmony": "#7f3c8d",
    "Timbre / Tone Colour": "#f58518",
    "Rhythm / Temporal": "#e45756",
    "Musical Texture": "#9d9d9d",
    "Musical Form": "#bdbdbd",
    "Expressivity": "#59a14f",
}


def parse_opensmile_feature_name(column: str) -> dict:
    """Separa agregação, funcional e derivada sem alterar a categoria acústica."""
    if column.startswith(MEAN_PREFIX):
        block_aggregation = "mean"
        source_feature = column[len(MEAN_PREFIX):]
    elif column.startswith(STD_PREFIX):
        block_aggregation = "std"
        source_feature = column[len(STD_PREFIX):]
    else:
        block_aggregation = "frame_level"
        source_feature = column

    functional_match = re.search(r"_(amean|stddev)$", source_feature, flags=re.IGNORECASE)
    source_functional = functional_match.group(1).lower() if functional_match else "none"
    without_functional = (
        source_feature[:functional_match.start()] if functional_match else source_feature
    )
    has_delta = bool(re.search(r"_de(?=\[|_|$)", without_functional, flags=re.IGNORECASE))
    acoustic_core = re.sub(
        r"_sma(?:_de)?(?=\[|_|$)", "", without_functional, flags=re.IGNORECASE
    )
    return {
        "coluna_cache": column,
        "feature_origem": source_feature,
        "nucleo_acustico": acoustic_core,
        "agregacao_bloco": block_aggregation,
        "funcional_origem": source_functional,
        "tem_derivada_de": has_delta,
    }


def classify_panda_opensmile(acoustic_core: str) -> Tuple[str, str]:
    """Classifica pela identidade acústica; `_de` e estatísticas não mudam a classe."""
    core = acoustic_core.lower()

    if "f0final" in core or "voicingfinalunclipped" in core:
        return "Pitch / Melody", "altura fundamental ou evidência local de vozeamento"
    if any(token in core for token in ["jitter", "shimmer", "hnr"]):
        return "Expressivity", "qualidade vocal, estabilidade e ruído harmônico"
    if any(token in core for token in ["rmsenergy", "lengthl1norm"]):
        return "Dynamics", "energia ou intensidade espectral"
    if any(token in core for token in [
        "mfcc", "audspec", "spectral", "psysharpness", "fband", "zcr"
    ]):
        return "Timbre / Tone Colour", "envoltória, distribuição ou forma espectral"

    return "Unclassified", "revisão manual necessária"


def build_opensmile_taxonomy_row(column: str) -> dict:
    row = parse_opensmile_feature_name(column)
    category, criterion = classify_panda_opensmile(row["nucleo_acustico"])
    row["categoria_panda"] = category
    row["criterio_classificacao"] = criterion
    return row


taxonomy_detail = pd.DataFrame(build_opensmile_taxonomy_row(c) for c in os_cols)
source_taxonomy = taxonomy_detail.drop_duplicates("feature_origem").copy()

taxonomy_summary = pd.DataFrame({"categoria_panda": PANDA_CATEGORY_ORDER})
source_counts = source_taxonomy["categoria_panda"].value_counts()
block_counts = taxonomy_detail["categoria_panda"].value_counts()
taxonomy_summary["descritores_frame_level"] = (
    taxonomy_summary["categoria_panda"].map(source_counts).fillna(0).astype(int)
)
taxonomy_summary["atributos_por_bloco"] = (
    taxonomy_summary["categoria_panda"].map(block_counts).fillna(0).astype(int)
)
taxonomy_summary["percentual_dos_520"] = (
    taxonomy_summary["atributos_por_bloco"] / len(taxonomy_detail) * 100
)

# Invariantes metodológicas: 261 colunas no CSV, 260 preditores e 520 atributos no bloco.
assert len(osdf_sample.columns) == 261, f"Esperadas 261 colunas no CSV; obtidas {len(osdf_sample.columns)}"
assert "frameTime" in osdf_sample.columns and "frameTime" not in feat_cols_ref
assert len(feat_cols_ref) == 260, f"Esperados 260 descritores openSMILE; obtidos {len(feat_cols_ref)}"
assert len(os_mean_cols) == 260 and len(os_std_cols) == 260
assert len(source_taxonomy) == 260 and len(taxonomy_detail) == 520
assert taxonomy_summary["descritores_frame_level"].sum() == 260
assert taxonomy_summary["atributos_por_bloco"].sum() == 520
assert not taxonomy_detail["categoria_panda"].eq("Unclassified").any()
assert set(c[len(MEAN_PREFIX):] for c in os_mean_cols) == set(
    c[len(STD_PREFIX):] for c in os_std_cols
)

mfcc_example = "os_std__pcm_fftMag_mfcc_sma_de[1]_amean"
mfcc_category = build_opensmile_taxonomy_row(mfcc_example)["categoria_panda"]
assert mfcc_category == "Timbre / Tone Colour"

taxonomy_detail_path = COMMON_DIR / "opensmile_taxonomy_features.csv"
taxonomy_summary_path = COMMON_DIR / "opensmile_taxonomy_summary.csv"
taxonomy_detail.to_csv(taxonomy_detail_path, index=False, encoding="utf-8-sig")
taxonomy_summary.to_csv(taxonomy_summary_path, index=False, encoding="utf-8-sig")

print("=== AUDITORIA TAXONÔMICA OPENSMILE ===")
print(f"Colunas no CSV de referência : {len(osdf_sample.columns)}")
print("Coluna temporal excluída     : frameTime")
print(f"Descritores frame-level      : {len(source_taxonomy)}")
print(f"Atributos por bloco          : {len(taxonomy_detail)} (260 mean + 260 std)")
print(f"Exemplo MFCC                 : {mfcc_example} -> {mfcc_category}")
print("\nTabela para revisão da Tabela 14:")
print(taxonomy_summary.to_string(index=False, formatters={"percentual_dos_520": "{:.2f}".format}))
print(f"\nDetalhamento salvo: {taxonomy_detail_path}")
print(f"Resumo salvo       : {taxonomy_summary_path}")

plot_taxonomy = taxonomy_summary[taxonomy_summary["atributos_por_bloco"] > 0].iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 4.8))
bars = ax.barh(
    plot_taxonomy["categoria_panda"],
    plot_taxonomy["atributos_por_bloco"],
    color=[PANDA_CATEGORY_COLORS[c] for c in plot_taxonomy["categoria_panda"]],
)
for bar, value in zip(bars, plot_taxonomy["atributos_por_bloco"]):
    ax.text(value + 4, bar.get_y() + bar.get_height() / 2, str(value), va="center")
ax.set_xlim(0, max(plot_taxonomy["atributos_por_bloco"]) * 1.15)
ax.set_xlabel("Atributos preditores por bloco")
ax.set_title("Taxonomia de Panda das 520 features openSMILE")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
taxonomy_figure_path = COMMON_DIR / "opensmile_taxonomy_panda.png"
plt.savefig(taxonomy_figure_path, bbox_inches="tight", dpi=160)
plt.show()
print(f"Figura salva       : {taxonomy_figure_path}")

=== AUDITORIA TAXONÔMICA OPENSMILE ===
Colunas no CSV de referência : 261
Coluna temporal excluída     : frameTime
Descritores frame-level      : 260
Atributos por bloco          : 520 (260 mean + 260 std)
Exemplo MFCC                 : os_std__pcm_fftMag_mfcc_sma_de[1]_amean -> Timbre / Tone Colour

Tabela para revisão da Tabela 14:
     categoria_panda  descritores_frame_level  atributos_por_bloco percentual_dos_520
      Pitch / Melody                        8                   16               3.08
            Dynamics                       12                   24               4.62
             Harmony                        0                    0               0.00
Timbre / Tone Colour                      224                  448              86.15
   Rhythm / Temporal                        0                    0               0.00
     Musical Texture                        0                    0               0.00
        Musical Form                        0                 

Figura salva       : Dados\comparative_outputs\common\opensmile_taxonomy_panda.png


## 7. Validar cobertura vs blocos canônicos

In [9]:
keys = [SONG_ID_COL, BLOCK_ID_COL, START_COL, END_COL]
blocks_expected = len(blocks)
blocks_got = len(result)

missing_songs = set(blocks[SONG_ID_COL].unique()) - set(result[SONG_ID_COL].unique())

print("=== VALIDAÇÃO DE COBERTURA ===")
print(f"Blocos esperados (fingerprint): {blocks_expected}")
print(f"Blocos no cache openSMILE    : {blocks_got}")
print(f"Cobertura                    : {blocks_got / blocks_expected * 100:.1f}%")

if missing_songs:
    print(f"\n⚠ {len(missing_songs)} músicas sem CSV openSMILE: {sorted(missing_songs)[:10]}")
else:
    print("\n✓ Todas as músicas têm CSV openSMILE.")

# Verificar alinhamento de valence/arousal
merged_check = result[[SONG_ID_COL, BLOCK_ID_COL, START_COL, VALENCE_COL, AROUSAL_COL]].merge(
    blocks[[SONG_ID_COL, BLOCK_ID_COL, START_COL, VALENCE_COL, AROUSAL_COL]],
    on=[SONG_ID_COL, BLOCK_ID_COL, START_COL],
    suffixes=("_cache", "_fp"),
)
v_diff = (merged_check["valence_cache"] - merged_check["valence_fp"]).abs().max()
a_diff = (merged_check["arousal_cache"] - merged_check["arousal_fp"]).abs().max()
print(f"\nVerificação labels: max|valence diff|={v_diff:.2e}  max|arousal diff|={a_diff:.2e}")
if v_diff < 1e-9 and a_diff < 1e-9:
    print("✓ Labels perfeitamente alinhados com fingerprint parquet.")
else:
    print("⚠ Pequena divergência (pode ser arredondamento de float).")

=== VALIDAÇÃO DE COBERTURA ===
Blocos esperados (fingerprint): 6506
Blocos no cache openSMILE    : 6506
Cobertura                    : 100.0%

✓ Todas as músicas têm CSV openSMILE.

Verificação labels: max|valence diff|=0.00e+00  max|arousal diff|=0.00e+00
✓ Labels perfeitamente alinhados com fingerprint parquet.


## 8. Distribuição dos frames por bloco (diagnóstico)

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma de n_frames_in_window
axes[0].hist(result["n_frames_in_window"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Frames openSMILE por bloco (janela de 10s)")
axes[0].set_xlabel("n_frames")
axes[0].set_ylabel("Blocos")
axes[0].axvline(result["n_frames_in_window"].median(), color="red",
                linestyle="--", label=f"Mediana={result['n_frames_in_window'].median():.0f}")
axes[0].legend()

# Scatter valence fingerprint vs valence openSMILE block (para confirmar alinhamento)
axes[1].scatter(
    merged_check["valence_fp"],
    merged_check["valence_cache"],
    alpha=0.2, s=3, color="seagreen"
)
axes[1].plot([-1, 1], [-1, 1], "r--", lw=0.8)
axes[1].set_title("Valence: fingerprint parquet vs cache")
axes[1].set_xlabel("fingerprint")
axes[1].set_ylabel("cache")

plt.tight_layout()
fig_path = COMMON_DIR / "opensmile_blocks_diagnostics.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=120)
plt.show()
print(f"Figura salva: {fig_path}")

Figura salva: Dados\comparative_outputs\common\opensmile_blocks_diagnostics.png


## 9. Salvar cache e inventário

In [11]:
# Remover coluna auxiliar de diagnóstico antes de salvar
result_save = result.drop(columns=["n_frames_in_window"])

result_save.to_parquet(OUTPUT_PATH, index=False)
size_mb = OUTPUT_PATH.stat().st_size / 1024 / 1024

os_mean_save = [c for c in result_save.columns if c.startswith(MEAN_PREFIX)]
os_std_save  = [c for c in result_save.columns if c.startswith(STD_PREFIX)]

print(f"Cache salvo: {OUTPUT_PATH}")
print(f"  Shape  : {result_save.shape}")
print(f"  Tamanho: {size_mb:.1f} MB")
print(f"  Colunas meta      : {SONG_ID_COL}, filename, {BLOCK_ID_COL}, {START_COL}, {END_COL},")
print(f"                       block_duration_sec, {VALENCE_COL}, {AROUSAL_COL}, quadrant, {QUADRANT_COL}, method")
print(f"  Features os_mean__: {len(os_mean_save)} (média dos frames no bloco)")
print(f"  Features os_std__ : {len(os_std_save)} (desvio padrão dos frames no bloco)")

# Inventário CSV para referência
inventory = pd.DataFrame({
    "coluna":    result_save.columns.tolist(),
    "tipo":      [str(result_save[c].dtype) for c in result_save.columns],
    "nan_count": [int(result_save[c].isna().sum()) for c in result_save.columns],
    "eh_feature": [c.startswith(FEATURE_PREFIX) for c in result_save.columns],
    "eh_mean":    [c.startswith(MEAN_PREFIX) for c in result_save.columns],
    "eh_std":     [c.startswith(STD_PREFIX)  for c in result_save.columns],
})
taxonomy_lookup = taxonomy_detail.set_index("coluna_cache")
inventory["feature_origem"] = inventory["coluna"].map(taxonomy_lookup["feature_origem"])
inventory["nucleo_acustico"] = inventory["coluna"].map(taxonomy_lookup["nucleo_acustico"])
inventory["agregacao_bloco"] = inventory["coluna"].map(taxonomy_lookup["agregacao_bloco"])
inventory["tem_derivada_de"] = inventory["coluna"].map(taxonomy_lookup["tem_derivada_de"])
inventory["categoria_panda"] = inventory["coluna"].map(taxonomy_lookup["categoria_panda"])
inv_path = COMMON_DIR / "opensmile_blocks_inventory.csv"
inventory.to_csv(inv_path, index=False, encoding="utf-8-sig")
print(f"\nInventário salvo: {inv_path}")

print("\n" + "=" * 65)
print("CACHE CONSTRUÍDO COM SUCESSO")
print("Execute agora os notebooks 01, 02 e 03 para os experimentos.")
print("=" * 65)

Cache salvo: Dados\comparative_outputs\common\opensmile_blocks.parquet
  Shape  : (6506, 531)
  Tamanho: 31.6 MB
  Colunas meta      : song_id, filename, block_id, block_start_sec, block_end_sec,
                       block_duration_sec, valence, arousal, quadrant, quadrant_label, method
  Features os_mean__: 260 (média dos frames no bloco)
  Features os_std__ : 260 (desvio padrão dos frames no bloco)

Inventário salvo: Dados\comparative_outputs\common\opensmile_blocks_inventory.csv

CACHE CONSTRUÍDO COM SUCESSO
Execute agora os notebooks 01, 02 e 03 para os experimentos.


## Verificação rápida do cache salvo

In [12]:
# Re-lê o cache para confirmar que foi salvo corretamente
check = pd.read_parquet(OUTPUT_PATH)
os_mean_check = [c for c in check.columns if c.startswith(MEAN_PREFIX)]
os_std_check  = [c for c in check.columns if c.startswith(STD_PREFIX)]

print("Verificação do arquivo salvo:")
print(f"  Shape          : {check.shape}")
print(f"  Features mean  : {len(os_mean_check)}")
print(f"  Features std   : {len(os_std_check)}")
print(f"  Músicas        : {check[SONG_ID_COL].nunique()}")
print(f"  NaN total      : {check.isna().sum().sum()}")
print(f"  Valence range  : [{check[VALENCE_COL].min():.3f}, {check[VALENCE_COL].max():.3f}]")
print(f"  Arousal range  : [{check[AROUSAL_COL].min():.3f}, {check[AROUSAL_COL].max():.3f}]")
print(f"  quadrant dist  :")
print(check["quadrant"].value_counts().to_string())
print(f"\n✓ Cache pronto para uso pelos notebooks 01, 02 e 03.")

Verificação do arquivo salvo:
  Shape          : (6506, 531)
  Features mean  : 260
  Features std   : 260
  Músicas        : 1802
  NaN total      : 0
  Valence range  : [-0.826, 0.703]
  Arousal range  : [-0.720, 0.891]
  quadrant dist  :
quadrant
Q1    3217
Q3    1382
Q2    1019
Q4     888

✓ Cache pronto para uso pelos notebooks 01, 02 e 03.
